# Imports

In [1]:

# Test converter compatibility
from models.base import BaseModel, extract_commands_and_params_from_string, get_tokenizer_and_model, reconstruct_string, tokenize_commands_and_params
import json
from pathlib import Path
from tqdm import tqdm

JSON_FILES = [
    "data/sl_data/train.json",
    "data/sl_data/val.json",
    "data/sl_data/test.json"
]


/home/alief/miniconda3/envs/n/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Tests

In [2]:


total = 0
success = 0
full_success = 0
results = []

for json_path in JSON_FILES:
    with open(json_path, "r") as f:
        data = json.load(f)

        for item in tqdm(data):
            total +=1
            seq_str = item['command_sequence']
            
            try:
                commands, params = extract_commands_and_params_from_string(seq_str)
                reconstructed_str = reconstruct_string(commands, params)
                success += 1
                if reconstructed_str == seq_str:
                    full_success += 1
                results.append((seq_str, commands, params, reconstructed_str))
            except Exception as e:
                print(f"Error processing sequence: {seq_str}")
                print(f"Error: {e}")
                
print(f"Successfully processed {success}/{total} sequences.")
print(f"Fully successful reconstructions: {full_success}/{total}")
example = results[0]
print("Example:")
print("Original String:", example[0])
print("Extracted Commands:", example[1])
print("Extracted Parameters:", example[2])
print("Reconstructed String:", example[3])

100%|██████████| 952/952 [00:00<00:00, 10659.05it/s]

Successfully processed 18929/18929 sequences.
Fully successful reconstructions: 0/18929
Example:
Original String: line,9,9 <curve_end> line,9,53 <curve_end> line,34,48 <curve_end> line,53,9 <curve_end> <loop_end> <face_end> <sketch_end> add,31,43,31,31,31,1,0,0,0,0,1,0,-1,0,30,48,48 <extrude_end>
Extracted Commands: ['line', 'line', 'line', 'line', 'add']
Extracted Parameters: [[ 9.  9.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
   0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
   0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
   0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 9. 53.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
   0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
   0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
   0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [34. 48.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
   0.  0

In [3]:
import torch
from torch.utils.data import Dataset
import json
from tqdm import tqdm

CMD_MAP = {
    "line": 0,
    "arc": 1,
    "circle": 2,
    "add": 3,
    "cut": 4,
    "intersect": 5
}

PARAM_MAP = {
    "line": 2,
    "arc": 4,
    "circle": 8,
    "add": 17,
    "cut": 17,
    "intersect": 17
}


class CADDataset(Dataset):
    def __init__(self, json_files):
        self.samples = []

        for json_path in json_files:
            with open(json_path, "r") as f:
                data = json.load(f)

            for item in tqdm(data, desc=f"Loading {json_path}"):
                seq_str = item["command_sequence"]
                inp_str = item["description"] 
                
                try:
                    tokenized_commands, tokenized_params = tokenize_commands_and_params(seq_str)
                    
                    self.samples.append({
                        "text": inp_str,  # or description if you prefer
                        "cmd_targets": torch.tensor(tokenized_commands, dtype=torch.long),
                        "param_targets": torch.tensor(tokenized_params, dtype=torch.float32)
                    })

                except Exception as e:
                    raise e
                    continue  # skip bad samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def cad_collate_fn(batch):
    batch_size = len(batch)

    # max sequence length
    max_L = max(item["cmd_targets"].shape[0] for item in batch)
    max_D = max(item["param_targets"].shape[1] for item in batch)

    # containers
    cmd_targets = torch.zeros((batch_size, max_L), dtype=torch.long)
    param_targets = torch.zeros((batch_size, max_L, max_D), dtype=torch.float32)

    texts = [text for item in batch for text in [item["text"]]]

    for i, item in enumerate(batch):
        L = item["cmd_targets"].shape[0]

        cmd_targets[i, :L] = item["cmd_targets"]
        param_targets[i, :L, :] = item["param_targets"]

    res = {
        "text": texts,
        "cmd_targets": cmd_targets,
        "param_targets": param_targets,
    }
    
    return res
    
from torch.utils.data import DataLoader



In [ ]:


import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DASDAS")
def train(model, 
          tokenizer, 
          criterion, 
            dataloader, epochs=50, batch_size=16, lr=1e-4):
    model.to(device)

    optimizer = AdamW(model.parameters(), lr=lr)

    results = []
    for epoch in range(epochs):
        model.train()

        total_loss = 0.0
        total_cmd_loss = 0.0
        total_param_loss = 0.0

        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}")

        for batch in pbar:
            texts = batch["text"]

            cmd_targets = batch["cmd_targets"].to(device)
            param_targets = batch["param_targets"].to(device)

            # ---- Tokenize ----
            enc = tokenizer(
                texts,
                padding=True,
                truncation=True,
                return_tensors="pt"
            )
            
            attention_mask = enc["attention_mask"].to(device)

            input_ids = enc["input_ids"].to(device)
            
            # Teacher forcing shift
            tgt_input  = cmd_targets[:, :-1]
            tgt_output = cmd_targets[:, 1:]

            param_input  = param_targets[:, :-1, :]
            param_output = param_targets[:, 1:, :]
            optimizer.zero_grad()
            
            # ---- Forward ----
            cmd_logits, param_preds = model(
                input_ids,
                tgt_input,
                attention_mask=attention_mask
            )

            # ---- Loss ----
            loss, cmd_loss, param_loss = criterion(
                cmd_logits,
                param_preds,
                tgt_output,      # ✅ shifted
                param_output     # ✅ shifted
            )
            

            # ---- Backprop ----
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # ---- Logging ----
            total_loss += loss.item()
            total_cmd_loss += cmd_loss.item()
            total_param_loss += param_loss.item()

            pbar.set_postfix({
                "loss": loss.item(),
                "cmd": cmd_loss.item(),
                "param": param_loss.item()
            })
    
    results.append({
                "epoch": epoch+1,
                "loss": total_loss / len(dataloader),
                "cmd": total_cmd_loss / len(dataloader),
                "param": total_param_loss / len(dataloader)
            })

DASDAS


# Runner

In [5]:
dataset = CADDataset(JSON_FILES)

dataloader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=cad_collate_fn,
    num_workers=2
)

tokenizer, model, criterion = get_tokenizer_and_model(max_len=max(item["cmd_targets"].shape[0] for item in dataset))
train(model, tokenizer, criterion, dataloader, epochs=50, batch_size=2, lr=1e-4)

Epoch 1:   0%|          | 0/2367 [00:00<?, ?it/s]We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1553: indexSelectLargeIndex: block: [248,0,0], thread: [96,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1553: indexSelectLargeIndex: block: [248,0,0], thread: [97,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1553: indexSelectLargeIndex: block: [248,0,0], thread: [98,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1553: indexSelectLargeIndex: block: [248,0,0], thread: [99,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1553: indexSelectLargeIndex: block: [248,0,0], thread: [100,0,0] As

torch.float32 torch.float32


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
